# Train an IMPSY model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cpmpercussion/impsy/blob/main/notebooks/impsy_training.ipynb)

This notebook trains an [IMPSY](https://charlesmartin.au/impsy) model from your `.log` files and gives you a `.tflite` model to perform with. Use it if you don't have a computer that can run IMPSY's training yourself.

Log files from IMPSY, IMPSYpi, the IMPSY AUv3 app and IMPSY Web all work. Their names end in `-{dimension}d-mdrnn.log`, e.g. `2026-09-01T12-00-00-9d-mdrnn.log`.

You don't need a GPU: training a small model on about 10 minutes of data takes a few minutes on Colab's standard CPU runtime.

Run each cell in order with the ▶ button (or Shift-Enter).

## 1. Install IMPSY

This takes a minute or two. If Colab asks you to restart the session afterwards, restart it and carry on from step 2.

In [ ]:
%pip install -q impsy

## 2. Upload your log files

Choose one or more `.log` files, or a `.zip` of them. Outside Colab, put your log files in a `logs` folder next to this notebook instead.

In [ ]:
import zipfile
from pathlib import Path

LOGS_DIR = Path("logs")
LOGS_DIR.mkdir(exist_ok=True)

try:
    from google.colab import files

    uploaded = files.upload()
    for name, content in uploaded.items():
        if name.endswith(".zip"):
            with zipfile.ZipFile(Path(name)) as z:
                for member in z.namelist():
                    if member.endswith(".log") and not member.startswith("__MACOSX"):
                        (LOGS_DIR / Path(member).name).write_bytes(z.read(member))
            Path(name).unlink()
        elif name.endswith(".log"):
            Path(name).rename(LOGS_DIR / Path(name).name)
        else:
            print(f"Skipping {name}: not a .log or .zip file")
except ImportError:
    print("Not running in Colab: using the log files already in", LOGS_DIR.resolve())

## 3. Check your logs

This counts your log files by dimension (the number of values in each log plus one for time). If you have logs of more than one dimension, it uses the one with the most files; set `DIMENSION` in the next step to choose a different one.

In [ ]:
import re
from collections import Counter

dimensions = Counter()
for log in sorted(LOGS_DIR.glob("*.log")):
    match = re.search(r"-(\d+)d-mdrnn\.log$", log.name)
    if match:
        dimensions[int(match.group(1))] += 1
    else:
        print(f"Ignoring {log.name}: its name should end in -{{dimension}}d-mdrnn.log")

for dim, count in sorted(dimensions.items()):
    print(f"{dim}d: {count} log file(s)")
assert dimensions, "No usable log files found, go back to step 2 and upload some."

## 4. Choose training settings

- **Model size:** `xs` or `s` is a good start. Bigger models (`m`, `l`, `xl`) can learn more but need much more data and take longer to train.
- **Max epochs:** training usually stops earlier, when the model stops improving for *patience* epochs.
- **Dimension:** leave at 0 to use the most common dimension from step 3.

In [ ]:
MODEL_SIZE = "s"  # @param ["xxs", "xs", "s", "m", "l", "xl"]
MAX_EPOCHS = 100  # @param {type:"integer"}
PATIENCE = 10  # @param {type:"integer"}
DIMENSION = 0  # @param {type:"integer"}

if DIMENSION == 0:
    DIMENSION = dimensions.most_common(1)[0][0]
assert DIMENSION in dimensions, (
    f"There are no {DIMENSION}d logs. Choose one of: "
    + ", ".join(f"{d}" for d in sorted(dimensions))
)
print(f"Training a {MODEL_SIZE} model on {DIMENSION}d data.")

## 5. Make a dataset

This collects all of your logs of the chosen dimension into one dataset file. It's good to have more than 10,000 interactions, but you can train on less. The model learns from runs of 50 events, so only logs with more than 51 events are used for training.

In [ ]:
import numpy as np
from impsy.dataset import generate_dataset
from impsy.train import SEQ_LEN

Path("datasets").mkdir(exist_ok=True)
dataset_file = generate_dataset(DIMENSION, source="logs", destination="datasets")
assert dataset_file is not None, (
    f"No data found in the {DIMENSION}d logs. Logs need rows recorded from "
    "your interface (with 'interface' in the second column)."
)

with np.load(dataset_file, allow_pickle=True) as loaded:
    long_enough = [p for p in loaded["perfs"] if len(p) > SEQ_LEN + 1]
print(f"{len(long_enough)} log(s) long enough to train on.")
assert long_enough, (
    f"None of your logs have more than {SEQ_LEN + 1} events, "
    "so there's nothing to train on. Record some longer logs."
)

## 6. Train

You'll see the loss for each epoch; lower is better. The `val_loss` is measured on data the model hasn't trained on, and training stops once it stops improving.

In [ ]:
from impsy.train import train_mdrnn

Path("models").mkdir(exist_ok=True)
output = train_mdrnn(
    DIMENSION,
    dataset_file,
    MODEL_SIZE,
    early_stopping=True,
    patience=PATIENCE,
    num_epochs=MAX_EPOCHS,
    batch_size=64,
    save_location="models",
)
tflite_file = Path(output["tflite_file"])
print("Trained model:", tflite_file)

## 7. Download your model

Download the `.tflite` file and copy it to your IMPSY setup:

- **IMPSY / IMPSYpi:** put it in the `models` folder (or upload it on the web UI's Models page) and set `file` under `[model]` in `config.toml`, along with the matching `dimension` and `size`.
- **IMPSY AUv3 and IMPSY Web:** load the `.tflite` file in the app.

In [ ]:
try:
    from google.colab import files

    files.download(str(tflite_file))
except ImportError:
    print("Your model is at", tflite_file.resolve())